# Swedish Address Fuzzy Matching Demo

This notebook demonstrates how to use the fuzzymatch library to match Swedish addresses.

## Setup

First, let's import the necessary modules and load our sample data.

In [ ]:
import sys
import pandas as pd

# Add parent directory to path to import our modules
sys.path.insert(0, '..')

from normalization import normalize_address, normalize_swedish_chars, extract_postal_code
from matching import fuzzy_match_address, find_best_match, similarity_score
from scoring import rank_matches, classify_match_quality, calculate_match_confidence
from evaluation import evaluate_score_distribution, calculate_match_rate

## Load Sample Data

Load the customer and lead address data.

In [ ]:
# Load customer and lead data
customers_df = pd.read_csv('../data/customers.csv')
leads_df = pd.read_csv('../data/leads.csv')

print(f"Loaded {len(customers_df)} customers and {len(leads_df)} leads")
print("\nSample customer data:")
print(customers_df.head())
print("\nSample lead data:")
print(leads_df.head())

## Address Normalization

Demonstrate Swedish address normalization features.

In [ ]:
# Example addresses
test_addresses = [
    "Drottninggatan 123",
    "Götgatan 45",
    "Östra Hamngatan 12",
    "  Stora   Gatan  78  "
]

print("Address Normalization Examples:")
print("=" * 60)
for addr in test_addresses:
    normalized = normalize_address(addr)
    print(f"Original:   '{addr}'")
    print(f"Normalized: '{normalized}'")
    print()

## Fuzzy Matching Examples

Match lead addresses against customer addresses.

In [ ]:
# Get all customer addresses
customer_addresses = customers_df['address'].tolist()

# Try to match a few lead addresses
sample_leads = leads_df.head(5)

print("Fuzzy Matching Results:")
print("=" * 80)

for idx, row in sample_leads.iterrows():
    lead_address = row['contact_address']
    print(f"\nLead Address: {lead_address}")
    
    # Find matches
    matches = fuzzy_match_address(lead_address, customer_addresses, score_cutoff=60.0, limit=3)
    
    if matches:
        print("Top Matches:")
        for match_addr, score in matches:
            quality = classify_match_quality(score)
            print(f"  - {match_addr:40} | Score: {score:.1f} | Quality: {quality}")
    else:
        print("  No matches found above threshold")

## Similarity Score Comparison

Compare different addresses and their similarity scores.

In [ ]:
# Compare similar addresses
address_pairs = [
    ("Drottninggatan 45", "Drottningsgatan 45"),
    ("Götgatan 45", "Götatan 45"),
    ("Kungsgatan 23", "Kungsvägen 23"),
    ("Östra Hamngatan 12", "Östra Hamnvägen 12"),
]

print("Similarity Score Comparisons:")
print("=" * 80)

for addr1, addr2 in address_pairs:
    score = similarity_score(addr1, addr2)
    quality = classify_match_quality(score)
    print(f"\nAddress 1: {addr1}")
    print(f"Address 2: {addr2}")
    print(f"Score:     {score:.2f} ({quality})")

## Batch Matching and Evaluation

Match all leads against customers and evaluate the results.

In [ ]:
# Match all leads
all_matches = []
all_scores = []

for idx, row in leads_df.iterrows():
    lead_address = row['contact_address']
    best_match = find_best_match(lead_address, customer_addresses, score_cutoff=60.0)
    
    if best_match:
        match_addr, score = best_match
        all_matches.append({
            'lead_id': row['lead_id'],
            'lead_address': lead_address,
            'matched_address': match_addr,
            'score': score,
            'quality': classify_match_quality(score)
        })
        all_scores.append(score)

# Create results dataframe
results_df = pd.DataFrame(all_matches)

print(f"\nMatched {len(results_df)} out of {len(leads_df)} leads")
print("\nSample Results:")
print(results_df.head(10))

## Match Quality Distribution

In [ ]:
# Analyze quality distribution
quality_counts = results_df['quality'].value_counts()

print("Match Quality Distribution:")
print("=" * 40)
for quality, count in quality_counts.items():
    percentage = (count / len(results_df)) * 100
    print(f"{quality:12} : {count:3} ({percentage:.1f}%)")

# Score statistics
score_stats = evaluate_score_distribution(all_scores)

print("\nScore Statistics:")
print("=" * 40)
for key, value in score_stats.items():
    print(f"{key:12} : {value:.2f}")

## Match Rate Calculation

In [ ]:
# Calculate match rate
match_rate_stats = calculate_match_rate(
    total_queries=len(leads_df),
    successful_matches=len(results_df),
    threshold=60.0
)

print("Match Rate Statistics:")
print("=" * 40)
print(f"Total Queries:      {match_rate_stats['total_queries']}")
print(f"Successful Matches: {match_rate_stats['successful_matches']}")
print(f"Match Rate:         {match_rate_stats['match_rate']:.2%}")
print(f"No Match Rate:      {match_rate_stats['no_match_rate']:.2%}")
print(f"Threshold:          {match_rate_stats['threshold']}")

## Save Results

In [ ]:
# Save matching results to CSV
output_path = '../data/matching_results.csv'
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")